# LTX-2 Text-to-Video Stages Analysis

**Pipeline Architecture:**
- **Stage 1**: Low-resolution diffusion (e.g., 544x960 @ 10 steps)
- **Stage 1.5**: ConvNet spatial upsampler (2x, deterministic)
- **Stage 2**: Full-resolution diffusion refinement (3 steps with distilled LoRA)

**This notebook:**
1. Runs the pipeline ONCE with stage capture
2. Decodes latents from each stage to video
3. Saves all three stage videos
4. Creates side-by-side comparison

In [ ]:
import sys
import rp
import torch
import numpy as np
from typing import Iterator

# Setup paths
IN_NOTEBOOK = rp.running_in_jupyter_notebook()
top_dir = rp.get_git_toplevel()
ltx_dir = rp.path_join(top_dir, 'LTX2')
ltx_src = rp.path_join(ltx_dir, 'src')
nfs_models_dir = rp.path_join(ltx_dir, 'models')
output_dir = rp.path_join(top_dir, 'outputs')

sys.path += [nfs_models_dir]
sys.path += rp.path_join(ltx_src, 'packages', ['ltx-core', 'ltx-trainer', 'ltx-pipelines'], 'src')

from download_models import local_download_dir, download_from_web
models_dir = local_download_dir

# LTX imports
from ltx_core.loader import LTXV_LORA_COMFY_RENAMING_MAP, LoraPathStrengthAndSDOps
from ltx_core.model.video_vae import TilingConfig, decode_video as vae_decode_video
from ltx_core.model.upsampler import upsample_video
from ltx_core.components.diffusion_steps import EulerDiffusionStep
from ltx_core.components.guiders import CFGGuider
from ltx_core.components.noisers import GaussianNoiser
from ltx_core.components.schedulers import LTX2Scheduler
from ltx_core.text_encoders.gemma import encode_text
from ltx_core.types import VideoPixelShape
from ltx_pipelines.ti2vid_two_stages import TI2VidTwoStagesPipeline
from ltx_pipelines.utils.helpers import (
    denoise_audio_video,
    euler_denoising_loop,
    guider_denoising_func,
    simple_denoising_func,
    image_conditionings_by_replacing_latent,
    cleanup_memory,
)
from ltx_pipelines.utils.constants import STAGE_2_DISTILLED_SIGMA_VALUES

rp.make_directory(output_dir)
rp.r._ensure_ffmpeg_installed()

print(f"Running in notebook: {IN_NOTEBOOK}")
print(f"Output directory: {output_dir}")

In [ ]:
# Model paths
checkpoint_path        = rp.path_join(models_dir, "ltx-2-19b-dev.safetensors")
distilled_lora_path    = rp.path_join(models_dir, "ltx-2-19b-distilled-lora-resized_dynamic_fro095_avg_rank_242_bf16.safetensors")
spatial_upsampler_path = rp.path_join(models_dir, "ltx-2-spatial-upscaler-x2-1.0.safetensors")
detailer_lora_path     = rp.path_join(models_dir, "ltx-2-19b-ic-lora-detailer.safetensors")
gemma_root             = models_dir

DEVICE = rp.select_torch_device(prefer_used=True, reserve=True)
DTYPE = torch.bfloat16

print(f"Device: {DEVICE}, dtype: {DTYPE}")
download_from_web()

In [ ]:
# Load pipeline
detailer_lora = LoraPathStrengthAndSDOps(detailer_lora_path, 1, LTXV_LORA_COMFY_RENAMING_MAP)
distilled_lora = LoraPathStrengthAndSDOps(distilled_lora_path, 1, LTXV_LORA_COMFY_RENAMING_MAP)

pipeline = TI2VidTwoStagesPipeline(
    checkpoint_path=checkpoint_path,
    distilled_lora=[distilled_lora, detailer_lora],
    spatial_upsampler_path=spatial_upsampler_path,
    gemma_root=gemma_root,
    loras=[detailer_lora],
)

print("Pipeline loaded!")

In [ ]:
# Generation parameters
prompt = '''EXT. NORTH ATLANTIC – OVERCAST DAY. Photorealistic Documentary Style. A wide, telephoto shot captures a rusted steel fishing trawler heaving violently in a rough sea. The lighting is flat and grey, diffused by thick storm clouds, creating a cold, desaturated palette. The ocean is deep green and churning with white foam. The boat plunges nose-first into a trough, sending a massive spray of white heavy mist over the bridge, then rises steeply as the buoyant bow cuts through the swell. The camera uses a "long lens" look, tracking the boat from a distance with slight handheld jitter, mimicking a cameraman on a chase boat trying to keep focus. Rain is visible as diagonal streaks against the grey sky. Audio of wind whipping the microphone and the rhythmic diesel chugging of the engine fighting the current.'''

negative_prompt = "worst quality, inconsistent motion, blurry, jittery, distorted, watermarks, low quality, artifacts, morphing, warping, flicker, text, logo"

prompt_slug = "motorboat_stormy_sea"

# Must be divisible by 64 for two-stage pipeline
height, width = 1088, 1920  # Stage 1 will be 544x960
num_frames = 121
frame_rate = 25.0
seed = 42
num_inference_steps = 10
cfg_guidance_scale = 4.0

print(f"Target: {height}x{width}, {num_frames} frames")
print(f"Stage 1 will be: {height//2}x{width//2}")
print(f"Stage 1.5/2 will be: {height}x{width}")

## Run Pipeline with Stage Capture

This runs the pipeline once and captures all three intermediate latents.

In [ ]:
def run_pipeline_with_stages(pipeline, prompt, negative_prompt, seed, height, width, 
                              num_frames, frame_rate, num_inference_steps, cfg_guidance_scale):
    """
    Run the two-stage pipeline and capture latents from all three stages.
    
    Returns:
        stage_1_latent: Latent after stage 1 diffusion (half resolution)
        stage_1_5_latent: Latent after ConvNet upsampling (full resolution)
        stage_2_latent: Latent after stage 2 diffusion (full resolution)
        audio_latent: Audio latent (final)
    """
    
    # Setup (from pipeline.__call__)
    generator = torch.Generator(device=pipeline.device).manual_seed(seed)
    noiser = GaussianNoiser(generator=generator)
    stepper = EulerDiffusionStep()
    cfg_guider = CFGGuider(cfg_guidance_scale)
    dtype = torch.bfloat16
    
    # Encode text
    print("Encoding text...")
    text_encoder = pipeline.stage_1_model_ledger.text_encoder()
    context_p, context_n = encode_text(text_encoder, prompts=[prompt, negative_prompt])
    v_context_p, a_context_p = context_p
    v_context_n, a_context_n = context_n
    
    torch.cuda.synchronize()
    del text_encoder
    cleanup_memory()
    
    # STAGE 1: Low-resolution diffusion
    print("\n=== STAGE 1: Low-resolution diffusion ===")
    video_encoder = pipeline.stage_1_model_ledger.video_encoder()
    transformer = pipeline.stage_1_model_ledger.transformer()
    sigmas = LTX2Scheduler().execute(steps=num_inference_steps).to(dtype=torch.float32, device=pipeline.device)
    
    def first_stage_denoising_loop(sigmas, video_state, audio_state, stepper):
        return euler_denoising_loop(
            sigmas=sigmas,
            video_state=video_state,
            audio_state=audio_state,
            stepper=stepper,
            denoise_fn=guider_denoising_func(
                cfg_guider, v_context_p, v_context_n, a_context_p, a_context_n, transformer=transformer
            ),
        )
    
    stage_1_output_shape = VideoPixelShape(
        batch=1,
        frames=num_frames,
        width=width // 2,
        height=height // 2,
        fps=frame_rate,
    )
    
    video_state, audio_state = denoise_audio_video(
        output_shape=stage_1_output_shape,
        conditionings=[],
        noiser=noiser,
        sigmas=sigmas,
        stepper=stepper,
        denoising_loop_fn=first_stage_denoising_loop,
        components=pipeline.pipeline_components,
        dtype=dtype,
        device=pipeline.device,
    )
    
    # CAPTURE STAGE 1 LATENT
    stage_1_latent = video_state.latent.clone()
    print(f"Stage 1 latent shape: {stage_1_latent.shape}")
    
    torch.cuda.synchronize()
    del transformer
    cleanup_memory()
    
    # STAGE 1.5: ConvNet upsampling (deterministic)
    print("\n=== STAGE 1.5: ConvNet spatial upsampling ===")
    upscaled_video_latent = upsample_video(
        latent=video_state.latent[:1],
        video_encoder=video_encoder,
        upsampler=pipeline.stage_2_model_ledger.spatial_upsampler(),
    )
    
    # CAPTURE STAGE 1.5 LATENT
    stage_1_5_latent = upscaled_video_latent.clone()
    print(f"Stage 1.5 latent shape: {stage_1_5_latent.shape}")
    
    torch.cuda.synchronize()
    cleanup_memory()
    
    # STAGE 2: Full-resolution diffusion refinement
    print("\n=== STAGE 2: Full-resolution diffusion refinement ===")
    transformer = pipeline.stage_2_model_ledger.transformer()
    distilled_sigmas = torch.Tensor(STAGE_2_DISTILLED_SIGMA_VALUES).to(pipeline.device)
    
    def second_stage_denoising_loop(sigmas, video_state, audio_state, stepper):
        return euler_denoising_loop(
            sigmas=sigmas,
            video_state=video_state,
            audio_state=audio_state,
            stepper=stepper,
            denoise_fn=simple_denoising_func(
                video_context=v_context_p,
                audio_context=a_context_p,
                transformer=transformer,
            ),
        )
    
    stage_2_output_shape = VideoPixelShape(
        batch=1, 
        frames=num_frames, 
        width=width, 
        height=height, 
        fps=frame_rate
    )
    
    video_state, audio_state = denoise_audio_video(
        output_shape=stage_2_output_shape,
        conditionings=[],
        noiser=noiser,
        sigmas=distilled_sigmas,
        stepper=stepper,
        denoising_loop_fn=second_stage_denoising_loop,
        components=pipeline.pipeline_components,
        dtype=dtype,
        device=pipeline.device,
        noise_scale=distilled_sigmas[0],
        initial_video_latent=upscaled_video_latent,
        initial_audio_latent=audio_state.latent,
    )
    
    # CAPTURE STAGE 2 LATENT
    stage_2_latent = video_state.latent.clone()
    audio_latent = audio_state.latent.clone()
    print(f"Stage 2 latent shape: {stage_2_latent.shape}")
    
    torch.cuda.synchronize()
    del transformer
    cleanup_memory()
    
    return stage_1_latent, stage_1_5_latent, stage_2_latent, audio_latent, video_encoder

# Run the pipeline with stage capture
with torch.inference_mode():
    stage_1_latent, stage_1_5_latent, stage_2_latent, audio_latent, video_encoder = run_pipeline_with_stages(
        pipeline=pipeline,
        prompt=prompt,
        negative_prompt=negative_prompt,
        seed=seed,
        height=height,
        width=width,
        num_frames=num_frames,
        frame_rate=frame_rate,
        num_inference_steps=num_inference_steps,
        cfg_guidance_scale=cfg_guidance_scale,
    )

print("\n=== Pipeline complete, all latents captured ===")

## Decode All Stages

Now decode each latent to video frames.

In [ ]:
def decode_and_save_stage(latent, stage_name, decoder, tiling_config, generator, 
                          prompt_slug, output_dir, frame_rate):
    """
    Decode a latent to video and save it.
    """
    print(f"\nDecoding {stage_name}...")
    
    # Decode latent to video
    video_frames = list(vae_decode_video(latent, decoder, tiling_config, generator))
    video_tensor = torch.cat(video_frames, dim=0)
    
    print(f"{stage_name} decoded shape: {video_tensor.shape}")
    
    # Save video
    resolution_str = f"{video_tensor.shape[1]}x{video_tensor.shape[2]}"
    filename = f"ltx2_t2v_{stage_name}_{resolution_str}_{prompt_slug}.mp4"
    path = rp.path_join(output_dir, filename)
    path = rp.get_unique_copy_path(path)
    
    video_np = rp.as_numpy_array(video_tensor.cpu())
    rp.save_video_mp4(video_np, path, framerate=frame_rate, video_bitrate=50000000)
    
    print(f"Saved: {path}")
    
    return video_np, path

# Decode all three stages
with torch.inference_mode():
    generator = torch.Generator(device=pipeline.device).manual_seed(seed)
    tiling_config = TilingConfig.default()
    
    # Get decoders
    decoder_stage1 = pipeline.stage_1_model_ledger.video_decoder()
    decoder_stage2 = pipeline.stage_2_model_ledger.video_decoder()
    
    # Decode Stage 1 (half resolution)
    video_s1, path_s1 = decode_and_save_stage(
        stage_1_latent, "stage1_lowres", decoder_stage1, tiling_config, generator,
        prompt_slug, output_dir, frame_rate
    )
    
    # Decode Stage 1.5 (full resolution, after upsampling)
    video_s1_5, path_s1_5 = decode_and_save_stage(
        stage_1_5_latent, "stage1.5_upsampled", decoder_stage2, tiling_config, generator,
        prompt_slug, output_dir, frame_rate
    )
    
    # Decode Stage 2 (full resolution, after refinement)
    video_s2, path_s2 = decode_and_save_stage(
        stage_2_latent, "stage2_final", decoder_stage2, tiling_config, generator,
        prompt_slug, output_dir, frame_rate
    )

print("\n=== All stages decoded and saved ===")

## Create Side-by-Side Comparison

In [ ]:
print("\nCreating comparison video...")

# Resize stage 1 to match others for comparison
video_s1_resized = rp.resize_images_to_fit(
    video_s1,
    height=video_s2.shape[1],
    width=video_s2.shape[2],
    allow_growth=True
)

# Create labeled videos
videos = [video_s1_resized, video_s1_5, video_s2]
labels = [
    f'Stage 1: {height//2}x{width//2}\n(Low-res diffusion)',
    f'Stage 1.5: {height}x{width}\n(ConvNet upsampled)',
    f'Stage 2: {height}x{width}\n(Refined diffusion)'
]

comparison_video = rp.horizontally_concatenated_videos(
    rp.resize_lists_to_min_len(
        rp.resize_videos_to_min_size(
            rp.labeled_videos(videos, labels, font='R:Futura')
        )
    )
)

# Save comparison
comparison_path = rp.path_join(output_dir, f"ltx2_stages_comparison_{prompt_slug}.mp4")
comparison_path = rp.get_unique_copy_path(comparison_path)
rp.save_video_mp4(comparison_video, comparison_path, framerate=frame_rate, video_bitrate=50000000)

print(f"Saved comparison: {comparison_path}")

if IN_NOTEBOOK:
    rp.display_video(comparison_path, framerate=frame_rate)

## Summary

In [ ]:
print("\n" + "="*80)
print("STAGE ANALYSIS COMPLETE")
print("="*80)

print(f"\nPrompt: {prompt[:100]}...")

print(f"\nGeneration Parameters:")
print(f"  Target resolution: {height}x{width}")
print(f"  Frames: {num_frames}")
print(f"  FPS: {frame_rate}")
print(f"  Seed: {seed}")
print(f"  Stage 1 steps: {num_inference_steps}")
print(f"  Stage 2 steps: {len(STAGE_2_DISTILLED_SIGMA_VALUES)}")
print(f"  CFG Scale: {cfg_guidance_scale}")

print(f"\nStage Outputs:")
print(f"  1. Stage 1 (Low-res diffusion):")
print(f"     {path_s1}")
print(f"     Resolution: {video_s1.shape[1]}x{video_s1.shape[2]}")
print(f"  2. Stage 1.5 (ConvNet upsampling):")
print(f"     {path_s1_5}")
print(f"     Resolution: {video_s1_5.shape[1]}x{video_s1_5.shape[2]}")
print(f"  3. Stage 2 (Diffusion refinement):")
print(f"     {path_s2}")
print(f"     Resolution: {video_s2.shape[1]}x{video_s2.shape[2]}")
print(f"  4. Side-by-side comparison:")
print(f"     {comparison_path}")

print("\n✓ All stages captured and saved successfully!")